In [ ]:
# ==============================================================================
# TRAINING SCRIPT: DISTILBERT BASE -> FINE-TUNED V1 (Manual Loop)
# Split: 70% Train - 15% Val - 15% Test
# ==============================================================================

# 1. Cài đặt thư viện
!pip install -q transformers torch scikit-learn pandas accelerate

import os
import json
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForMultipleChoice, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from google.colab import drive

# 2. Kết nối Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive', force_remount=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[INFO] Device: {DEVICE}")

# --- CẤU HÌNH  ---
DATA_FILE = '/content/drive/MyDrive/data MCQ 9K.jsonl'
BASE_MODEL_NAME = 'distilbert-base-uncased'
OUTPUT_DIR = '/content/drive/MyDrive/DistilBERT_Retrained_QA_Model_V1_MultipleChoice'
TEST_FILE_OUTPUT = '/content/drive/MyDrive/distilbert_test_data_v1.jsonl'

BATCH_SIZE = 16
EPOCHS = 4
LEARNING_RATE = 5e-5
MAX_LEN = 256

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# ============================================================
# 1. LOAD DATA & PROCESS (GIỮ NGUYÊN LOGIC BẠN MUỐN)
# ============================================================
def load_data(file_path):
    print(f"[INFO] Reading dataset: {os.path.basename(file_path)}")
    data = []
    if not os.path.exists(file_path): return []

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                if line.strip().startswith('['):
                    items = json.loads(line)
                    if isinstance(items, list):
                        for obj in items: process_item(obj, data)
                else:
                    obj = json.loads(line)
                    process_item(obj, data)
            except: continue

    print(f"   -> Loaded {len(data)} samples.")
    return data

def process_item(obj, data_list):
    try:
        q = obj.get('question')
        # Hỗ trợ lấy từ nhiều key khác nhau để tránh lỗi
        opts = obj.get('options') or obj.get('choices') or obj.get('raw_options')
        ans = obj.get('answer') or obj.get('label') or obj.get('correct_answer') or obj.get('raw_answer')

        if q and opts and ans is not None: # check ans is not None vì 0 là False
            choices = []
            answer_idx = -1

            # Xử lý nếu options là Dict {'A':..., 'B':...}
            if isinstance(opts, dict):
                sorted_k = sorted(opts.keys())
                choices = [str(opts[k]) for k in sorted_k]
                # Map đáp án chữ cái sang số
                if isinstance(ans, str) and ans in sorted_k:
                     answer_idx = sorted_k.index(ans)

            # Xử lý nếu options là List [..., ...]
            elif isinstance(opts, list):
                choices = [str(o) for o in opts]
                if isinstance(ans, str) and len(ans) == 1:
                    char_map = {'A':0, 'B':1, 'C':2, 'D':3}
                    answer_idx = char_map.get(ans.upper(), -1)
                elif isinstance(ans, int):
                    answer_idx = ans

            # Kiểm tra hợp lệ và padding
            if len(choices) >= 2 and answer_idx != -1 and answer_idx < 4:
                while len(choices) < 4: choices.append("-")
                choices = choices[:4]
                data_list.append({"question": str(q), "choices": choices, "label": int(answer_idx)})
    except Exception as e:
        pass

# ============================================================
# 2. SPLIT DATA (70 - 15 - 15)
# ============================================================
print("\n[INFO] Preparing Data & Splitting...")
all_data = load_data(DATA_FILE)
if not all_data:
    print("❌ Không đọc được dữ liệu!")
    import sys; sys.exit()

# BƯỚC 1: Tách Test (15%)
train_val_data, test_data = train_test_split(all_data, test_size=0.15, random_state=42)

# BƯỚC 2: Tách Val (15% TỔNG) -> Tính tỷ lệ trên tập còn lại
val_split_ratio = 0.15 / 0.85
train_data, val_data = train_test_split(train_val_data, test_size=val_split_ratio, random_state=42)

print(f"   📊 Tổng cộng: {len(all_data)}")
print(f"   🔹 Train (70%): {len(train_data)} câu")
print(f"   🔹 Val   (15%): {len(val_data)} câu")
print(f"   🔹 Test  (15%): {len(test_data)} câu")

# Lưu file Test
with open(TEST_FILE_OUTPUT, 'w', encoding='utf-8') as f:
    for item in test_data:
        save_obj = {
            "question": item['question'],
            "choices": item['choices'],
            "label": item['label']
        }
        f.write(json.dumps(save_obj, ensure_ascii=False) + '\n')

# ============================================================
# 3. SETUP MODEL & TRAINING (DISTILBERT)
# ============================================================
print(f"\n[INFO] Loading Base Model: {BASE_MODEL_NAME}")

# Load Tokenizer & Model DistilBERT
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
model = AutoModelForMultipleChoice.from_pretrained(BASE_MODEL_NAME).to(DEVICE)

class QADataset(Dataset):
    def __init__(self, tokenizer, data_list, max_len=256):
        self.tokenizer = tokenizer
        self.data = data_list
        self.max_len = max_len

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        # DistilBERT xử lý Input giống PhoBERT: [Question] * 4 ghép với [Choices]
        # Tạo 4 cặp (Question, Option A), (Question, Option B)...
        inputs = self.tokenizer(
            [item["question"]] * 4,
            item["choices"],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        # Reshape inputs cho đúng chuẩn (4, seq_len)
        return {
            "input_ids": inputs["input_ids"], # Shape [4, 256]
            "attention_mask": inputs["attention_mask"],
            "labels": torch.tensor(item["label"], dtype=torch.long)
        }

train_loader = DataLoader(QADataset(tokenizer, train_data), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(QADataset(tokenizer, val_data), batch_size=BATCH_SIZE, shuffle=False)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
scheduler = get_linear_schedule_with_warmup(optimizer, 0, len(train_loader) * EPOCHS)

# ============================================================
# 4. TRAINING LOOP
# ============================================================
print("\n" + "="*40 + "\n🚀 START TRAINING DISTILBERT\n" + "="*40)
best_loss = float('inf')

for epoch in range(EPOCHS):
    model.train()
    print(f"\nEpoch {epoch + 1}/{EPOCHS}")
    prog_bar = tqdm(train_loader, desc="Training")
    total_loss = 0

    for batch in prog_bar:
        optimizer.zero_grad()

        # Đưa dữ liệu vào GPU (Batch shape: [batch_size, 4, seq_len])
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)

        # DistilBERT Forward pass
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        prog_bar.set_postfix({'loss': loss.item()})

    # --- Validation ---
    model.eval()
    val_loss = 0; correct = 0; total = 0
    print("   -> Validating...")
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            val_loss += outputs.loss.item()

            # Tính accuracy
            preds = torch.argmax(outputs.logits, dim=1)
            correct += (preds == labels).sum().item()
            total += len(labels)

    avg_loss = val_loss / len(val_loader)
    acc = correct / total
    print(f"   📊 Train Loss: {total_loss/len(train_loader):.4f} | Val Loss: {avg_loss:.4f} | Val Acc: {acc:.4f}")

    # Lưu model tốt nhất
    if avg_loss < best_loss:
        best_loss = avg_loss
        print(f"   🔥 Saving Best Model to {OUTPUT_DIR}...")
        model.save_pretrained(OUTPUT_DIR)
        tokenizer.save_pretrained(OUTPUT_DIR)

print("\n🏆 TRAINING COMPLETE!")

In [ ]:
# ==============================================================================
# GENERIC EVALUATION SCRIPT (CHẤM THI DISTILBERT MULTIPLE CHOICE)
# ==============================================================================

# 1. Cài đặt thư viện
# !pip install -q transformers torch scikit-learn pandas accelerate

import os
import json
import torch
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForMultipleChoice
from sklearn.metrics import accuracy_score, classification_report
from tqdm import tqdm
from google.colab import drive

# 2. Kết nối Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive', force_remount=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[INFO] Device: {DEVICE}")

# --- CẤU HÌNH (SỬA Ở ĐÂY CHO DISTILBERT) ---

# 1. Đường dẫn Model DistilBERT cần chấm
CURRENT_MODEL_PATH = '/content/drive/MyDrive/DistilBERT_FineTuned_QA_Model_V1_MultipleChoice'

# 2. File dữ liệu đề thi (Test Set - JSONL)
TEST_DATA_PATH = '/content/drive/MyDrive/data MCQ 9K.jsonl'

# 3. Nơi lưu kết quả (File chứa các câu làm sai) -> ĐÃ CẬP NHẬT THEO YÊU CẦU
OUTPUT_WRONG_ANSWERS = '/content/drive/MyDrive/Benchmark_Wrong_Cases/wrong_answers_V1.jsonl'

# Tham số khác
BATCH_SIZE = 16
MAX_LEN = 256

# ============================================================
# 1. HÀM ĐỌC DỮ LIỆU
# ============================================================
def load_data(file_path):
    print(f"[INFO] Reading data from: {os.path.basename(file_path)}")
    data = []
    if not os.path.exists(file_path):
        print(f"[ERROR] File not found: {file_path}")
        return []

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                obj = json.loads(line)
                if 'question' in obj and 'options' in obj and 'answer' in obj:
                    opts = obj['options']
                    sorted_k = sorted(opts.keys()) # A, B, C, D

                    if obj['answer'] in sorted_k:
                        data.append({
                            "question": obj['question'],
                            "choices": [opts[k] for k in sorted_k],
                            "label": sorted_k.index(obj['answer']),
                            "raw_options": opts,
                            "raw_answer": obj['answer']
                        })
            except: continue

    print(f"[INFO] Loaded {len(data)} valid samples.")
    return data

# ============================================================
# 2. DATASET CLASS
# ============================================================
class EvalDataset(Dataset):
    def __init__(self, tokenizer, data_list, max_len=256):
        self.tokenizer = tokenizer
        self.data = data_list
        self.max_len = max_len

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        inputs = self.tokenizer(
            [item["question"]] * len(item["choices"]),
            item["choices"],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids": inputs["input_ids"],
            "attention_mask": inputs["attention_mask"],
            "labels": torch.tensor(item["label"], dtype=torch.long)
        }

# ============================================================
# 3. HÀM CHẤM THI (MAIN EVALUATION)
# ============================================================
def run_evaluation():
    # Load Data
    all_data = load_data(TEST_DATA_PATH)
    if not all_data: return

    # Load Model & Tokenizer
    print(f"\n[INFO] Loading model from: {CURRENT_MODEL_PATH}")
    try:
        try:
            tokenizer = AutoTokenizer.from_pretrained(CURRENT_MODEL_PATH)
        except:
            print("[WARN] Local tokenizer not found. Using fallback: 'distilbert-base-multilingual-cased'")
            tokenizer = AutoTokenizer.from_pretrained("distilbert-base-multilingual-cased", use_fast=True)

        model = AutoModelForMultipleChoice.from_pretrained(CURRENT_MODEL_PATH).to(DEVICE)
        model.eval()
        print("[INFO] Model loaded successfully.")

    except Exception as e:
        print(f"[ERROR] Critical failure loading model: {e}")
        return

    # Prepare Loader
    dataset = EvalDataset(tokenizer, all_data, max_len=MAX_LEN)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

    print("\n[INFO] Starting Scoring Process for DistilBERT...")

    all_preds = []
    all_labels = []
    wrong_cases = []
    batch_start_idx = 0

    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            targets = labels.cpu().numpy()

            all_preds.extend(preds)
            all_labels.extend(targets)

            # Ghi lại câu sai
            for i in range(len(preds)):
                if preds[i] != targets[i]:
                    original_item = all_data[batch_start_idx + i]
                    char_map = ['A', 'B', 'C', 'D']
                    pred_char = char_map[preds[i]] if preds[i] < 4 else "Unknown"

                    wrong_cases.append({
                        "question": original_item['question'],
                        "options": original_item['raw_options'],
                        "correct_answer": original_item['raw_answer'],
                        "model_prediction": pred_char
                    })

            batch_start_idx += len(preds)

    # Calculate Metrics
    acc = accuracy_score(all_labels, all_preds)

    print("\n" + "="*40)
    print(f"REPORT RESULTS")
    print("="*40)
    print(f"Model Path:      {os.path.basename(CURRENT_MODEL_PATH)}")
    print(f"Total Questions: {len(all_data)}")
    print(f"Correct:         {len(all_data) - len(wrong_cases)}")
    print(f"Wrong:           {len(wrong_cases)}")
    print(f"Accuracy:        {acc:.4f} ({acc*100:.2f}%)")

    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds, target_names=['A', 'B', 'C', 'D'], zero_division=0))

    # Save Wrong Answers
    if wrong_cases:
        # Tự động tạo thư mục nếu chưa tồn tại (Benchmark_Wrong_Cases)
        output_dir = os.path.dirname(OUTPUT_WRONG_ANSWERS)
        if output_dir and not os.path.exists(output_dir):
            print(f"[INFO] Creating directory: {output_dir}")
            os.makedirs(output_dir)

        with open(OUTPUT_WRONG_ANSWERS, 'w', encoding='utf-8') as f:
            for item in wrong_cases:
                f.write(json.dumps(item, ensure_ascii=False) + '\n')
        print(f"\n[SAVE] Saved {len(wrong_cases)} wrong answers to: {OUTPUT_WRONG_ANSWERS}")
    else:
        print("\n[INFO] Perfect Score! No wrong answers file generated.")

if __name__ == "__main__":
    run_evaluation()

In [ ]:

# ==============================================================================

# TOOL SINH DỮ LIỆU GROQ (PHIÊN BẢN TỐC ĐỘ CAO - 8B INSTANT)

# ==============================================================================



!pip install -q groq tqdm



import os

import json

import time

from groq import Groq, RateLimitError, APIError, BadRequestError

from google.colab import drive

from tqdm import tqdm



# 1. Kết nối Drive

if not os.path.exists('/content/drive'):

    drive.mount('/content/drive', force_remount=True)



# --- CẤU HÌNH ---

GROQ_API_KEY = "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"



# 🔥 QUAN TRỌNG: CHUYỂN SANG 8B ĐỂ CÓ QUOTA CAO HƠN

GROQ_MODEL = "qwen/qwen3-32b"



INPUT_FILE = '/content/drive/MyDrive/DistilBERT/Benchmark_Wrong_Cases/wrong_answers_V1.jsonl'

OUTPUT_FILE = '/content/drive/MyDrive/DistilBERT/Generated_data_distil/generated_data_groq_distil_v1.jsonl'



# 🔥 GIẢM BATCH XUỐNG 1 ĐỂ AN TOÀN TUYỆT ĐỐI VỚI RATE LIMIT

BATCH_SIZE = 5

NUM_VARIANTS = 4



client = Groq(api_key=GROQ_API_KEY)



# ============================================================

# HÀM GỌI API

# ============================================================

def get_groq_response(batch_content):



    system_prompt = "You are an Expert Exam Creator. You output JSON only."



    # Prompt được tối ưu cho 8B

    user_prompt = f"""

    Task: Create exactly {NUM_VARIANTS} NEW multiple-choice variants for the input question.



    INPUT:

    {json.dumps(batch_content, ensure_ascii=False)}



    OUTPUT JSON FORMAT:

    {{

      "questions": [

        {{ "question": "...", "options": {{ "A": "...", "B": "...", "C": "...", "D": "..." }}, "answer": "A" }},

        ...

      ]

    }}

    STRICTLY JSON. NO EXPLANATION.

    """



    max_retries = 10

    for attempt in range(max_retries):

        try:

            chat_completion = client.chat.completions.create(

                messages=[

                    {"role": "system", "content": system_prompt},

                    {"role": "user", "content": user_prompt}

                ],

                model=GROQ_MODEL,

                response_format={"type": "json_object"},

                temperature=0.6,

                max_tokens=2048 # Giới hạn output để tiết kiệm TPM

            )

            return chat_completion.choices[0].message.content



        except RateLimitError:

            wait_time = 10 + (attempt * 5) # Chờ tăng dần: 10s, 15s, 20s...

            print(f"\r   ⏳ Hết Quota 8B. Đang nghỉ {wait_time}s... (Lần {attempt+1})", end="")

            time.sleep(wait_time)



        except (APIError, BadRequestError) as e:

            print(f"\n   ⚠️ Lỗi API: {e}. Thử lại sau 2s...")

            time.sleep(2)

        except Exception as e:

            print(f"\n   ⚠️ Lỗi lạ: {e}")

            time.sleep(2)



    return None



# ============================================================

# MAIN LOOP

# ============================================================

def main():

    print(f"🚀 BẮT ĐẦU VỚI MODEL: {GROQ_MODEL} (Tốc độ cao)")



    if not os.path.exists(INPUT_FILE):

        print(f"❌ Lỗi: Không tìm thấy file {INPUT_FILE}")

        return



    input_data = []

    with open(INPUT_FILE, 'r', encoding='utf-8') as f:

        for line in f:

            try: input_data.append(json.loads(line))

            except: continue



    total_input = len(input_data)

    print(f"📚 Tổng số câu sai cần xử lý: {total_input}")



    # Resume Logic

    start_idx = 0

    if os.path.exists(OUTPUT_FILE):

        with open(OUTPUT_FILE, 'r') as f:

            existing_lines = sum(1 for _ in f)



        # Tính toán lại vị trí resume

        processed_items = existing_lines // NUM_VARIANTS

        start_idx = processed_items



        # Căn chỉnh batch

        start_idx = (start_idx // BATCH_SIZE) * BATCH_SIZE

        print(f"🔄 File cũ có {existing_lines} dòng. Tiếp tục từ câu gốc thứ {start_idx}...")



    print("\n⚡ Đang chạy...")



    with open(OUTPUT_FILE, 'a', encoding='utf-8') as f_out:



        # Cắt danh sách input

        remaining_data = input_data[start_idx:]

        batches = [remaining_data[i:i + BATCH_SIZE] for i in range(0, len(remaining_data), BATCH_SIZE)]



        for batch in tqdm(batches, desc="Tiến độ"):



            # Chuẩn bị Input

            batch_content = []

            for item in batch:

                q = item.get('question') or item.get('q')

                opts = item.get('options') or item.get('raw_options') or item.get('opts')

                ans = item.get('answer') or item.get('raw_answer') or item.get('correct_answer')

                if q and opts and ans:

                    batch_content.append({"q": q, "opts": opts, "ans": ans})



            if not batch_content: continue



            # Gọi Groq

            json_str = get_groq_response(batch_content)



            if json_str:

                try:

                    data = json.loads(json_str)

                    new_qs = []



                    if isinstance(data, dict):

                        # Ưu tiên tìm key "questions"

                        if "questions" in data and isinstance(data["questions"], list):

                            new_qs = data["questions"]

                        else:

                            # Tìm bất kỳ key nào chứa list

                            for key in data:

                                if isinstance(data[key], list):

                                    new_qs = data[key]

                                    break

                    elif isinstance(data, list):

                        new_qs = data



                    for q in new_qs:

                        if 'question' in q and 'options' in q:

                            f_out.write(json.dumps(q, ensure_ascii=False) + "\n")



                    f_out.flush()

                    os.fsync(f_out.fileno())



                except json.JSONDecodeError:

                    print("   ❌ Lỗi Parse JSON")



            # Model 8B rất nhanh, nghỉ cực ngắn là đủ

            time.sleep(0.5)



    print("\n" + "="*50)

    print("✅ HOÀN TẤT QUÁ TRÌNH!")

    print(f"📂 File kết quả: {OUTPUT_FILE}")



if __name__ == "__main__":

  main()

In [ ]:
!pip install -q transformers torch scikit-learn pandas accelerate

import os
import json
import torch
import random
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForMultipleChoice, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from google.colab import drive

drive.mount('/content/drive', force_remount=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Device: {DEVICE}")

# --- CẤU HÌNH ---
# Model cũ (V2)
OLD_MODEL_PATH = '/content/drive/MyDrive/DistilBERT_Retrained_QA_Model_V1_MultipleChoice'

# Dữ liệu
FILE_ORIGINAL = '/content/drive/MyDrive/data MCQ 9K.jsonl'
FILE_GENERATED = '/content/drive/MyDrive/generated_data_v1.jsonl'

# Model mới (V3)
NEW_MODEL_OUTPUT_DIR = '/content/drive/MyDrive/DistilBERT_Retrained_QA_Model_V2_MultipleChoice'

BATCH_SIZE = 16
EPOCHS = 4
LR = 3e-5

if not os.path.exists(NEW_MODEL_OUTPUT_DIR):
    os.makedirs(NEW_MODEL_OUTPUT_DIR)

# ============================================================
# PHẦN 1: HÀM ĐỌC DỮ LIỆU
# ============================================================
def load_data(file_path, tag="Data"):
    print(f"📂 Reading {tag}: {os.path.basename(file_path)}...")
    data = []

    if not os.path.exists(file_path):
        print(f"   ⚠️ File not found: {file_path}")
        return []

    count = 0
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                if line.strip().startswith('['):
                    items = json.loads(line)
                    if isinstance(items, list):
                        for obj in items:
                            if process_item(obj, data): count += 1
                else:
                    obj = json.loads(line)
                    if process_item(obj, data): count += 1
            except: continue

    print(f"   -> Loaded: {count}")
    return data

def process_item(obj, data_list):
    try:
        q = obj.get('question')
        opts = obj.get('options') or obj.get('choices') or obj.get('raw_options')
        ans = obj.get('answer') or obj.get('raw_answer') or obj.get('correct_answer') or obj.get('true_answer') or obj.get('label')

        if q and opts and ans is not None:
            choices = []
            answer_idx = -1

            if isinstance(opts, dict):
                sorted_k = sorted(opts.keys())
                choices = [str(opts[k]) for k in sorted_k]
                if isinstance(ans, str) and ans in sorted_k: answer_idx = sorted_k.index(ans)
            elif isinstance(opts, list):
                choices = [str(o) for o in opts]
                if isinstance(ans, str) and len(ans) == 1:
                    char_map = {'A':0, 'B':1, 'C':2, 'D':3}
                    answer_idx = char_map.get(ans.upper(), -1)
                elif isinstance(ans, int):
                    answer_idx = ans

            if len(choices) >= 2 and answer_idx != -1 and answer_idx < 4:
                while len(choices) < 4: choices.append("-")
                choices = choices[:4]

                data_list.append({
                    "question": str(q),
                    "choices": choices,
                    "label": int(answer_idx)
                })
                return True
    except: pass
    return False

# ============================================================
# PHẦN 2: CHUẨN BỊ TRAIN
# ============================================================
print("\n" + "="*40)
print("🥣 PREPARING DATA")
print("="*40)

data_orig = load_data(FILE_ORIGINAL, "Original")
data_gen = load_data(FILE_GENERATED, "Generated")

final_train_data = data_orig + data_gen
random.shuffle(final_train_data)

print(f"\n📚 Total samples: {len(final_train_data)}")

if len(final_train_data) == 0:
    import sys; sys.exit("❌ No data found!")

print(f"\n📥 Loading Model: {OLD_MODEL_PATH}")
try:
    tokenizer = AutoTokenizer.from_pretrained(OLD_MODEL_PATH)
    model = AutoModelForMultipleChoice.from_pretrained(OLD_MODEL_PATH).to(DEVICE)
    print("✅ Loaded Pre-trained Model")
except:
    print("⚠️ Pre-trained not found. Loading Base DistilBERT...")
    tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
    model = AutoModelForMultipleChoice.from_pretrained("distilbert-base-uncased").to(DEVICE)

class QADataset(Dataset):
    def __init__(self, tokenizer, data_list, max_len=256):
        self.tokenizer = tokenizer
        self.data = data_list
        self.max_len = max_len

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        inputs = self.tokenizer(
            [item["question"]] * 4,
            item["choices"],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids": inputs["input_ids"],
            "attention_mask": inputs["attention_mask"],
            "labels": torch.tensor(item["label"], dtype=torch.long)
        }

train_set, val_set = train_test_split(final_train_data, test_size=0.1, random_state=42)
train_loader = DataLoader(QADataset(tokenizer, train_set, 256), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(QADataset(tokenizer, val_set, 256), batch_size=BATCH_SIZE, shuffle=False)

optimizer = AdamW(model.parameters(), lr=LR)
scheduler = get_linear_schedule_with_warmup(optimizer, 0, len(train_loader) * EPOCHS)

# ============================================================
# PHẦN 3: HUẤN LUYỆN
# ============================================================
print("\n" + "="*40)
print("🚀 START TRAINING")
print("="*40)

best_loss = float('inf')

for epoch in range(EPOCHS):
    model.train()
    print(f"\nEpoch {epoch + 1}/{EPOCHS}")

    prog_bar = tqdm(train_loader, desc="Training")
    total_loss = 0

    for batch in prog_bar:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)

        input_ids = input_ids.view(-1, 4, input_ids.shape[-1])
        attention_mask = attention_mask.view(-1, 4, attention_mask.shape[-1])

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        prog_bar.set_postfix({'loss': loss.item()})

    model.eval()
    val_loss = 0; correct = 0; total = 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)

            input_ids = input_ids.view(-1, 4, input_ids.shape[-1])
            attention_mask = attention_mask.view(-1, 4, attention_mask.shape[-1])

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            val_loss += outputs.loss.item()
            preds = torch.argmax(outputs.logits, dim=1)
            correct += (preds == labels).sum().item()
            total += len(labels)

    avg_loss = val_loss / len(val_loader)
    acc = correct / total
    print(f"   -> Val Loss: {avg_loss:.4f} | Val Accuracy: {acc:.4f}")

    if avg_loss < best_loss:
        best_loss = avg_loss
        print(f"🔥 Saving Best Model to: {NEW_MODEL_OUTPUT_DIR}")
        model.save_pretrained(NEW_MODEL_OUTPUT_DIR)
        tokenizer.save_pretrained(NEW_MODEL_OUTPUT_DIR)

print("\n🏆 TRAINING COMPLETED!")

In [ ]:
# ==============================================================================
# GENERIC EVALUATION SCRIPT (CHẤM THI DISTILBERT MULTIPLE CHOICE)
# ==============================================================================

# 1. Cài đặt thư viện
# !pip install -q transformers torch scikit-learn pandas accelerate

import os
import json
import torch
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForMultipleChoice
from sklearn.metrics import accuracy_score, classification_report
from tqdm import tqdm
from google.colab import drive

# 2. Kết nối Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive', force_remount=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[INFO] Device: {DEVICE}")

# --- CẤU HÌNH (SỬA Ở ĐÂY CHO DISTILBERT) ---

# 1. Đường dẫn Model DistilBERT cần chấm
CURRENT_MODEL_PATH = '/content/drive/MyDrive/DistilBERT_FineTuned_QA_Model_V2_MultipleChoice'

# 2. File dữ liệu đề thi (Test Set - JSONL)
TEST_DATA_PATH = '/content/drive/MyDrive/data MCQ 9K.jsonl'

# 3. Nơi lưu kết quả (File chứa các câu làm sai) -> ĐÃ CẬP NHẬT THEO YÊU CẦU
OUTPUT_WRONG_ANSWERS = '/content/drive/MyDrive/Benchmark_Wrong_Cases/wrong_answers_V2.jsonl'

# Tham số khác
BATCH_SIZE = 16
MAX_LEN = 256

# ============================================================
# 1. HÀM ĐỌC DỮ LIỆU
# ============================================================
def load_data(file_path):
    print(f"[INFO] Reading data from: {os.path.basename(file_path)}")
    data = []
    if not os.path.exists(file_path):
        print(f"[ERROR] File not found: {file_path}")
        return []

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                obj = json.loads(line)
                if 'question' in obj and 'options' in obj and 'answer' in obj:
                    opts = obj['options']
                    sorted_k = sorted(opts.keys()) # A, B, C, D

                    if obj['answer'] in sorted_k:
                        data.append({
                            "question": obj['question'],
                            "choices": [opts[k] for k in sorted_k],
                            "label": sorted_k.index(obj['answer']),
                            "raw_options": opts,
                            "raw_answer": obj['answer']
                        })
            except: continue

    print(f"[INFO] Loaded {len(data)} valid samples.")
    return data

# ============================================================
# 2. DATASET CLASS
# ============================================================
class EvalDataset(Dataset):
    def __init__(self, tokenizer, data_list, max_len=256):
        self.tokenizer = tokenizer
        self.data = data_list
        self.max_len = max_len

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        inputs = self.tokenizer(
            [item["question"]] * len(item["choices"]),
            item["choices"],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids": inputs["input_ids"],
            "attention_mask": inputs["attention_mask"],
            "labels": torch.tensor(item["label"], dtype=torch.long)
        }

# ============================================================
# 3. HÀM CHẤM THI (MAIN EVALUATION)
# ============================================================
def run_evaluation():
    # Load Data
    all_data = load_data(TEST_DATA_PATH)
    if not all_data: return

    # Load Model & Tokenizer
    print(f"\n[INFO] Loading model from: {CURRENT_MODEL_PATH}")
    try:
        try:
            tokenizer = AutoTokenizer.from_pretrained(CURRENT_MODEL_PATH)
        except:
            print("[WARN] Local tokenizer not found. Using fallback: 'distilbert-base-multilingual-cased'")
            tokenizer = AutoTokenizer.from_pretrained("distilbert-base-multilingual-cased", use_fast=True)

        model = AutoModelForMultipleChoice.from_pretrained(CURRENT_MODEL_PATH).to(DEVICE)
        model.eval()
        print("[INFO] Model loaded successfully.")

    except Exception as e:
        print(f"[ERROR] Critical failure loading model: {e}")
        return

    # Prepare Loader
    dataset = EvalDataset(tokenizer, all_data, max_len=MAX_LEN)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

    print("\n[INFO] Starting Scoring Process for DistilBERT...")

    all_preds = []
    all_labels = []
    wrong_cases = []
    batch_start_idx = 0

    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            targets = labels.cpu().numpy()

            all_preds.extend(preds)
            all_labels.extend(targets)

            # Ghi lại câu sai
            for i in range(len(preds)):
                if preds[i] != targets[i]:
                    original_item = all_data[batch_start_idx + i]
                    char_map = ['A', 'B', 'C', 'D']
                    pred_char = char_map[preds[i]] if preds[i] < 4 else "Unknown"

                    wrong_cases.append({
                        "question": original_item['question'],
                        "options": original_item['raw_options'],
                        "correct_answer": original_item['raw_answer'],
                        "model_prediction": pred_char
                    })

            batch_start_idx += len(preds)

    # Calculate Metrics
    acc = accuracy_score(all_labels, all_preds)

    print("\n" + "="*40)
    print(f"REPORT RESULTS")
    print("="*40)
    print(f"Model Path:      {os.path.basename(CURRENT_MODEL_PATH)}")
    print(f"Total Questions: {len(all_data)}")
    print(f"Correct:         {len(all_data) - len(wrong_cases)}")
    print(f"Wrong:           {len(wrong_cases)}")
    print(f"Accuracy:        {acc:.4f} ({acc*100:.2f}%)")

    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds, target_names=['A', 'B', 'C', 'D'], zero_division=0))

    # Save Wrong Answers
    if wrong_cases:
        # Tự động tạo thư mục nếu chưa tồn tại (Benchmark_Wrong_Cases)
        output_dir = os.path.dirname(OUTPUT_WRONG_ANSWERS)
        if output_dir and not os.path.exists(output_dir):
            print(f"[INFO] Creating directory: {output_dir}")
            os.makedirs(output_dir)

        with open(OUTPUT_WRONG_ANSWERS, 'w', encoding='utf-8') as f:
            for item in wrong_cases:
                f.write(json.dumps(item, ensure_ascii=False) + '\n')
        print(f"\n[SAVE] Saved {len(wrong_cases)} wrong answers to: {OUTPUT_WRONG_ANSWERS}")
    else:
        print("\n[INFO] Perfect Score! No wrong answers file generated.")

if __name__ == "__main__":
    run_evaluation()

In [ ]:

# ==============================================================================

# TOOL SINH DỮ LIỆU GROQ (PHIÊN BẢN TỐC ĐỘ CAO - 8B INSTANT)

# ==============================================================================



!pip install -q groq tqdm



import os

import json

import time

from groq import Groq, RateLimitError, APIError, BadRequestError

from google.colab import drive

from tqdm import tqdm



# 1. Kết nối Drive

if not os.path.exists('/content/drive'):

    drive.mount('/content/drive', force_remount=True)



# --- CẤU HÌNH ---

GROQ_API_KEY = "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"



# 🔥 QUAN TRỌNG: CHUYỂN SANG 8B ĐỂ CÓ QUOTA CAO HƠN

GROQ_MODEL = "qwen/qwen3-32b"



INPUT_FILE = '/content/drive/MyDrive/DistilBERT/Benchmark_Wrong_Cases/wrong_answers_QA_Model_V2.jsonl'

OUTPUT_FILE = '/content/drive/MyDrive/DistilBERT/Generated_data_distil/generated_data_groq_distil_v2.jsonl'



# 🔥 GIẢM BATCH XUỐNG 1 ĐỂ AN TOÀN TUYỆT ĐỐI VỚI RATE LIMIT

BATCH_SIZE = 5

NUM_VARIANTS = 4



client = Groq(api_key=GROQ_API_KEY)



# ============================================================

# HÀM GỌI API

# ============================================================

def get_groq_response(batch_content):



    system_prompt = "You are an Expert Exam Creator. You output JSON only."



    # Prompt được tối ưu cho 8B

    user_prompt = f"""

    Task: Create exactly {NUM_VARIANTS} NEW multiple-choice variants for the input question.



    INPUT:

    {json.dumps(batch_content, ensure_ascii=False)}



    OUTPUT JSON FORMAT:

    {{

      "questions": [

        {{ "question": "...", "options": {{ "A": "...", "B": "...", "C": "...", "D": "..." }}, "answer": "A" }},

        ...

      ]

    }}

    STRICTLY JSON. NO EXPLANATION.

    """



    max_retries = 10

    for attempt in range(max_retries):

        try:

            chat_completion = client.chat.completions.create(

                messages=[

                    {"role": "system", "content": system_prompt},

                    {"role": "user", "content": user_prompt}

                ],

                model=GROQ_MODEL,

                response_format={"type": "json_object"},

                temperature=0.6,

                max_tokens=2048 # Giới hạn output để tiết kiệm TPM

            )

            return chat_completion.choices[0].message.content



        except RateLimitError:

            wait_time = 10 + (attempt * 5) # Chờ tăng dần: 10s, 15s, 20s...

            print(f"\r   ⏳ Hết Quota 8B. Đang nghỉ {wait_time}s... (Lần {attempt+1})", end="")

            time.sleep(wait_time)



        except (APIError, BadRequestError) as e:

            print(f"\n   ⚠️ Lỗi API: {e}. Thử lại sau 2s...")

            time.sleep(2)

        except Exception as e:

            print(f"\n   ⚠️ Lỗi lạ: {e}")

            time.sleep(2)



    return None



# ============================================================

# MAIN LOOP

# ============================================================

def main():

    print(f"🚀 BẮT ĐẦU VỚI MODEL: {GROQ_MODEL} (Tốc độ cao)")



    if not os.path.exists(INPUT_FILE):

        print(f"❌ Lỗi: Không tìm thấy file {INPUT_FILE}")

        return



    input_data = []

    with open(INPUT_FILE, 'r', encoding='utf-8') as f:

        for line in f:

            try: input_data.append(json.loads(line))

            except: continue



    total_input = len(input_data)

    print(f"📚 Tổng số câu sai cần xử lý: {total_input}")



    # Resume Logic

    start_idx = 0

    if os.path.exists(OUTPUT_FILE):

        with open(OUTPUT_FILE, 'r') as f:

            existing_lines = sum(1 for _ in f)



        # Tính toán lại vị trí resume

        processed_items = existing_lines // NUM_VARIANTS

        start_idx = processed_items



        # Căn chỉnh batch

        start_idx = (start_idx // BATCH_SIZE) * BATCH_SIZE

        print(f"🔄 File cũ có {existing_lines} dòng. Tiếp tục từ câu gốc thứ {start_idx}...")



    print("\n⚡ Đang chạy...")



    with open(OUTPUT_FILE, 'a', encoding='utf-8') as f_out:



        # Cắt danh sách input

        remaining_data = input_data[start_idx:]

        batches = [remaining_data[i:i + BATCH_SIZE] for i in range(0, len(remaining_data), BATCH_SIZE)]



        for batch in tqdm(batches, desc="Tiến độ"):



            # Chuẩn bị Input

            batch_content = []

            for item in batch:

                q = item.get('question') or item.get('q')

                opts = item.get('options') or item.get('raw_options') or item.get('opts')

                ans = item.get('answer') or item.get('raw_answer') or item.get('correct_answer')

                if q and opts and ans:

                    batch_content.append({"q": q, "opts": opts, "ans": ans})



            if not batch_content: continue



            # Gọi Groq

            json_str = get_groq_response(batch_content)



            if json_str:

                try:

                    data = json.loads(json_str)

                    new_qs = []



                    if isinstance(data, dict):

                        # Ưu tiên tìm key "questions"

                        if "questions" in data and isinstance(data["questions"], list):

                            new_qs = data["questions"]

                        else:

                            # Tìm bất kỳ key nào chứa list

                            for key in data:

                                if isinstance(data[key], list):

                                    new_qs = data[key]

                                    break

                    elif isinstance(data, list):

                        new_qs = data



                    for q in new_qs:

                        if 'question' in q and 'options' in q:

                            f_out.write(json.dumps(q, ensure_ascii=False) + "\n")



                    f_out.flush()

                    os.fsync(f_out.fileno())



                except json.JSONDecodeError:

                    print("   ❌ Lỗi Parse JSON")



            # Model 8B rất nhanh, nghỉ cực ngắn là đủ

            time.sleep(0.5)



    print("\n" + "="*50)

    print("✅ HOÀN TẤT QUÁ TRÌNH!")

    print(f"📂 File kết quả: {OUTPUT_FILE}")



if __name__ == "__main__":

  main()

In [ ]:
!pip install -q transformers torch scikit-learn pandas accelerate

import os
import json
import torch
import random
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForMultipleChoice, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from google.colab import drive

drive.mount('/content/drive', force_remount=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Device: {DEVICE}")

# --- CẤU HÌNH ---
# Model cũ (V2)
OLD_MODEL_PATH = '/content/drive/MyDrive/DistilBERT_Retrained_QA_Model_V2_MultipleChoice'

# Dữ liệu
FILE_ORIGINAL = '/content/drive/MyDrive/data MCQ 9K.jsonl'
FILE_GENERATED = '/content/drive/MyDrive/generated_data_v2.jsonl'

# Model mới (V3)
NEW_MODEL_OUTPUT_DIR = '/content/drive/MyDrive/DistilBERT_Retrained_QA_Model_V3_MultipleChoice'

BATCH_SIZE = 16
EPOCHS = 4
LR = 3e-5

if not os.path.exists(NEW_MODEL_OUTPUT_DIR):
    os.makedirs(NEW_MODEL_OUTPUT_DIR)

# ============================================================
# PHẦN 1: HÀM ĐỌC DỮ LIỆU
# ============================================================
def load_data(file_path, tag="Data"):
    print(f"📂 Reading {tag}: {os.path.basename(file_path)}...")
    data = []

    if not os.path.exists(file_path):
        print(f"   ⚠️ File not found: {file_path}")
        return []

    count = 0
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                if line.strip().startswith('['):
                    items = json.loads(line)
                    if isinstance(items, list):
                        for obj in items:
                            if process_item(obj, data): count += 1
                else:
                    obj = json.loads(line)
                    if process_item(obj, data): count += 1
            except: continue

    print(f"   -> Loaded: {count}")
    return data

def process_item(obj, data_list):
    try:
        q = obj.get('question')
        opts = obj.get('options') or obj.get('choices') or obj.get('raw_options')
        ans = obj.get('answer') or obj.get('raw_answer') or obj.get('correct_answer') or obj.get('true_answer') or obj.get('label')

        if q and opts and ans is not None:
            choices = []
            answer_idx = -1

            if isinstance(opts, dict):
                sorted_k = sorted(opts.keys())
                choices = [str(opts[k]) for k in sorted_k]
                if isinstance(ans, str) and ans in sorted_k: answer_idx = sorted_k.index(ans)
            elif isinstance(opts, list):
                choices = [str(o) for o in opts]
                if isinstance(ans, str) and len(ans) == 1:
                    char_map = {'A':0, 'B':1, 'C':2, 'D':3}
                    answer_idx = char_map.get(ans.upper(), -1)
                elif isinstance(ans, int):
                    answer_idx = ans

            if len(choices) >= 2 and answer_idx != -1 and answer_idx < 4:
                while len(choices) < 4: choices.append("-")
                choices = choices[:4]

                data_list.append({
                    "question": str(q),
                    "choices": choices,
                    "label": int(answer_idx)
                })
                return True
    except: pass
    return False

# ============================================================
# PHẦN 2: CHUẨN BỊ TRAIN
# ============================================================
print("\n" + "="*40)
print("🥣 PREPARING DATA")
print("="*40)

data_orig = load_data(FILE_ORIGINAL, "Original")
data_gen = load_data(FILE_GENERATED, "Generated")

final_train_data = data_orig + data_gen
random.shuffle(final_train_data)

print(f"\n📚 Total samples: {len(final_train_data)}")

if len(final_train_data) == 0:
    import sys; sys.exit("❌ No data found!")

print(f"\n📥 Loading Model: {OLD_MODEL_PATH}")
try:
    tokenizer = AutoTokenizer.from_pretrained(OLD_MODEL_PATH)
    model = AutoModelForMultipleChoice.from_pretrained(OLD_MODEL_PATH).to(DEVICE)
    print("✅ Loaded Pre-trained Model")
except:
    print("⚠️ Pre-trained not found. Loading Base DistilBERT...")
    tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
    model = AutoModelForMultipleChoice.from_pretrained("distilbert-base-uncased").to(DEVICE)

class QADataset(Dataset):
    def __init__(self, tokenizer, data_list, max_len=256):
        self.tokenizer = tokenizer
        self.data = data_list
        self.max_len = max_len

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        inputs = self.tokenizer(
            [item["question"]] * 4,
            item["choices"],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids": inputs["input_ids"],
            "attention_mask": inputs["attention_mask"],
            "labels": torch.tensor(item["label"], dtype=torch.long)
        }

train_set, val_set = train_test_split(final_train_data, test_size=0.1, random_state=42)
train_loader = DataLoader(QADataset(tokenizer, train_set, 256), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(QADataset(tokenizer, val_set, 256), batch_size=BATCH_SIZE, shuffle=False)

optimizer = AdamW(model.parameters(), lr=LR)
scheduler = get_linear_schedule_with_warmup(optimizer, 0, len(train_loader) * EPOCHS)

# ============================================================
# PHẦN 3: HUẤN LUYỆN
# ============================================================
print("\n" + "="*40)
print("🚀 START TRAINING")
print("="*40)

best_loss = float('inf')

for epoch in range(EPOCHS):
    model.train()
    print(f"\nEpoch {epoch + 1}/{EPOCHS}")

    prog_bar = tqdm(train_loader, desc="Training")
    total_loss = 0

    for batch in prog_bar:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)

        input_ids = input_ids.view(-1, 4, input_ids.shape[-1])
        attention_mask = attention_mask.view(-1, 4, attention_mask.shape[-1])

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        prog_bar.set_postfix({'loss': loss.item()})

    model.eval()
    val_loss = 0; correct = 0; total = 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)

            input_ids = input_ids.view(-1, 4, input_ids.shape[-1])
            attention_mask = attention_mask.view(-1, 4, attention_mask.shape[-1])

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            val_loss += outputs.loss.item()
            preds = torch.argmax(outputs.logits, dim=1)
            correct += (preds == labels).sum().item()
            total += len(labels)

    avg_loss = val_loss / len(val_loader)
    acc = correct / total
    print(f"   -> Val Loss: {avg_loss:.4f} | Val Accuracy: {acc:.4f}")

    if avg_loss < best_loss:
        best_loss = avg_loss
        print(f"🔥 Saving Best Model to: {NEW_MODEL_OUTPUT_DIR}")
        model.save_pretrained(NEW_MODEL_OUTPUT_DIR)
        tokenizer.save_pretrained(NEW_MODEL_OUTPUT_DIR)

print("\n🏆 TRAINING COMPLETED!")

In [ ]:
# ==============================================================================
# GENERIC EVALUATION SCRIPT (CHẤM THI DISTILBERT MULTIPLE CHOICE)
# ==============================================================================

# 1. Cài đặt thư viện
# !pip install -q transformers torch scikit-learn pandas accelerate

import os
import json
import torch
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForMultipleChoice
from sklearn.metrics import accuracy_score, classification_report
from tqdm import tqdm
from google.colab import drive

# 2. Kết nối Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive', force_remount=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[INFO] Device: {DEVICE}")

# --- CẤU HÌNH (SỬA Ở ĐÂY CHO DISTILBERT) ---

# 1. Đường dẫn Model DistilBERT cần chấm
CURRENT_MODEL_PATH = '/content/drive/MyDrive/DistilBERT_FineTuned_QA_Model_V3_MultipleChoice'

# 2. File dữ liệu đề thi (Test Set - JSONL)
TEST_DATA_PATH = '/content/drive/MyDrive/data MCQ 9K.jsonl'

# 3. Nơi lưu kết quả (File chứa các câu làm sai) -> ĐÃ CẬP NHẬT THEO YÊU CẦU
OUTPUT_WRONG_ANSWERS = '/content/drive/MyDrive/Benchmark_Wrong_Cases/wrong_answers_V3.jsonl'

# Tham số khác
BATCH_SIZE = 16
MAX_LEN = 256

# ============================================================
# 1. HÀM ĐỌC DỮ LIỆU
# ============================================================
def load_data(file_path):
    print(f"[INFO] Reading data from: {os.path.basename(file_path)}")
    data = []
    if not os.path.exists(file_path):
        print(f"[ERROR] File not found: {file_path}")
        return []

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                obj = json.loads(line)
                if 'question' in obj and 'options' in obj and 'answer' in obj:
                    opts = obj['options']
                    sorted_k = sorted(opts.keys()) # A, B, C, D

                    if obj['answer'] in sorted_k:
                        data.append({
                            "question": obj['question'],
                            "choices": [opts[k] for k in sorted_k],
                            "label": sorted_k.index(obj['answer']),
                            "raw_options": opts,
                            "raw_answer": obj['answer']
                        })
            except: continue

    print(f"[INFO] Loaded {len(data)} valid samples.")
    return data

# ============================================================
# 2. DATASET CLASS
# ============================================================
class EvalDataset(Dataset):
    def __init__(self, tokenizer, data_list, max_len=256):
        self.tokenizer = tokenizer
        self.data = data_list
        self.max_len = max_len

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        inputs = self.tokenizer(
            [item["question"]] * len(item["choices"]),
            item["choices"],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids": inputs["input_ids"],
            "attention_mask": inputs["attention_mask"],
            "labels": torch.tensor(item["label"], dtype=torch.long)
        }

# ============================================================
# 3. HÀM CHẤM THI (MAIN EVALUATION)
# ============================================================
def run_evaluation():
    # Load Data
    all_data = load_data(TEST_DATA_PATH)
    if not all_data: return

    # Load Model & Tokenizer
    print(f"\n[INFO] Loading model from: {CURRENT_MODEL_PATH}")
    try:
        try:
            tokenizer = AutoTokenizer.from_pretrained(CURRENT_MODEL_PATH)
        except:
            print("[WARN] Local tokenizer not found. Using fallback: 'distilbert-base-multilingual-cased'")
            tokenizer = AutoTokenizer.from_pretrained("distilbert-base-multilingual-cased", use_fast=True)

        model = AutoModelForMultipleChoice.from_pretrained(CURRENT_MODEL_PATH).to(DEVICE)
        model.eval()
        print("[INFO] Model loaded successfully.")

    except Exception as e:
        print(f"[ERROR] Critical failure loading model: {e}")
        return

    # Prepare Loader
    dataset = EvalDataset(tokenizer, all_data, max_len=MAX_LEN)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

    print("\n[INFO] Starting Scoring Process for DistilBERT...")

    all_preds = []
    all_labels = []
    wrong_cases = []
    batch_start_idx = 0

    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            targets = labels.cpu().numpy()

            all_preds.extend(preds)
            all_labels.extend(targets)

            # Ghi lại câu sai
            for i in range(len(preds)):
                if preds[i] != targets[i]:
                    original_item = all_data[batch_start_idx + i]
                    char_map = ['A', 'B', 'C', 'D']
                    pred_char = char_map[preds[i]] if preds[i] < 4 else "Unknown"

                    wrong_cases.append({
                        "question": original_item['question'],
                        "options": original_item['raw_options'],
                        "correct_answer": original_item['raw_answer'],
                        "model_prediction": pred_char
                    })

            batch_start_idx += len(preds)

    # Calculate Metrics
    acc = accuracy_score(all_labels, all_preds)

    print("\n" + "="*40)
    print(f"REPORT RESULTS")
    print("="*40)
    print(f"Model Path:      {os.path.basename(CURRENT_MODEL_PATH)}")
    print(f"Total Questions: {len(all_data)}")
    print(f"Correct:         {len(all_data) - len(wrong_cases)}")
    print(f"Wrong:           {len(wrong_cases)}")
    print(f"Accuracy:        {acc:.4f} ({acc*100:.2f}%)")

    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds, target_names=['A', 'B', 'C', 'D'], zero_division=0))

    # Save Wrong Answers
    if wrong_cases:
        # Tự động tạo thư mục nếu chưa tồn tại (Benchmark_Wrong_Cases)
        output_dir = os.path.dirname(OUTPUT_WRONG_ANSWERS)
        if output_dir and not os.path.exists(output_dir):
            print(f"[INFO] Creating directory: {output_dir}")
            os.makedirs(output_dir)

        with open(OUTPUT_WRONG_ANSWERS, 'w', encoding='utf-8') as f:
            for item in wrong_cases:
                f.write(json.dumps(item, ensure_ascii=False) + '\n')
        print(f"\n[SAVE] Saved {len(wrong_cases)} wrong answers to: {OUTPUT_WRONG_ANSWERS}")
    else:
        print("\n[INFO] Perfect Score! No wrong answers file generated.")

if __name__ == "__main__":
    run_evaluation()

In [ ]:

# ==============================================================================

# TOOL SINH DỮ LIỆU GROQ (PHIÊN BẢN TỐC ĐỘ CAO - 8B INSTANT)

# ==============================================================================



!pip install -q groq tqdm



import os

import json

import time

from groq import Groq, RateLimitError, APIError, BadRequestError

from google.colab import drive

from tqdm import tqdm



# 1. Kết nối Drive

if not os.path.exists('/content/drive'):

    drive.mount('/content/drive', force_remount=True)



# --- CẤU HÌNH ---

GROQ_API_KEY = "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"



# 🔥 QUAN TRỌNG: CHUYỂN SANG 8B ĐỂ CÓ QUOTA CAO HƠN

GROQ_MODEL = "qwen/qwen3-32b"



INPUT_FILE = '/content/drive/MyDrive/DistilBERT/Benchmark_Wrong_Cases/wrong_answers_QA_Model_V3.jsonl'

OUTPUT_FILE = '/content/drive/MyDrive/DistilBERT/Generated_data_distil/generated_data_groq_distil_v3.jsonl'



# 🔥 GIẢM BATCH XUỐNG 1 ĐỂ AN TOÀN TUYỆT ĐỐI VỚI RATE LIMIT

BATCH_SIZE = 5

NUM_VARIANTS = 4



client = Groq(api_key=GROQ_API_KEY)



# ============================================================

# HÀM GỌI API

# ============================================================

def get_groq_response(batch_content):



    system_prompt = "You are an Expert Exam Creator. You output JSON only."



    # Prompt được tối ưu cho 8B

    user_prompt = f"""

    Task: Create exactly {NUM_VARIANTS} NEW multiple-choice variants for the input question.



    INPUT:

    {json.dumps(batch_content, ensure_ascii=False)}



    OUTPUT JSON FORMAT:

    {{

      "questions": [

        {{ "question": "...", "options": {{ "A": "...", "B": "...", "C": "...", "D": "..." }}, "answer": "A" }},

        ...

      ]

    }}

    STRICTLY JSON. NO EXPLANATION.

    """



    max_retries = 10

    for attempt in range(max_retries):

        try:

            chat_completion = client.chat.completions.create(

                messages=[

                    {"role": "system", "content": system_prompt},

                    {"role": "user", "content": user_prompt}

                ],

                model=GROQ_MODEL,

                response_format={"type": "json_object"},

                temperature=0.6,

                max_tokens=2048 # Giới hạn output để tiết kiệm TPM

            )

            return chat_completion.choices[0].message.content



        except RateLimitError:

            wait_time = 10 + (attempt * 5) # Chờ tăng dần: 10s, 15s, 20s...

            print(f"\r   ⏳ Hết Quota 8B. Đang nghỉ {wait_time}s... (Lần {attempt+1})", end="")

            time.sleep(wait_time)



        except (APIError, BadRequestError) as e:

            print(f"\n   ⚠️ Lỗi API: {e}. Thử lại sau 2s...")

            time.sleep(2)

        except Exception as e:

            print(f"\n   ⚠️ Lỗi lạ: {e}")

            time.sleep(2)



    return None



# ============================================================

# MAIN LOOP

# ============================================================

def main():

    print(f"🚀 BẮT ĐẦU VỚI MODEL: {GROQ_MODEL} (Tốc độ cao)")



    if not os.path.exists(INPUT_FILE):

        print(f"❌ Lỗi: Không tìm thấy file {INPUT_FILE}")

        return



    input_data = []

    with open(INPUT_FILE, 'r', encoding='utf-8') as f:

        for line in f:

            try: input_data.append(json.loads(line))

            except: continue



    total_input = len(input_data)

    print(f"📚 Tổng số câu sai cần xử lý: {total_input}")



    # Resume Logic

    start_idx = 0

    if os.path.exists(OUTPUT_FILE):

        with open(OUTPUT_FILE, 'r') as f:

            existing_lines = sum(1 for _ in f)



        # Tính toán lại vị trí resume

        processed_items = existing_lines // NUM_VARIANTS

        start_idx = processed_items



        # Căn chỉnh batch

        start_idx = (start_idx // BATCH_SIZE) * BATCH_SIZE

        print(f"🔄 File cũ có {existing_lines} dòng. Tiếp tục từ câu gốc thứ {start_idx}...")



    print("\n⚡ Đang chạy...")



    with open(OUTPUT_FILE, 'a', encoding='utf-8') as f_out:



        # Cắt danh sách input

        remaining_data = input_data[start_idx:]

        batches = [remaining_data[i:i + BATCH_SIZE] for i in range(0, len(remaining_data), BATCH_SIZE)]



        for batch in tqdm(batches, desc="Tiến độ"):



            # Chuẩn bị Input

            batch_content = []

            for item in batch:

                q = item.get('question') or item.get('q')

                opts = item.get('options') or item.get('raw_options') or item.get('opts')

                ans = item.get('answer') or item.get('raw_answer') or item.get('correct_answer')

                if q and opts and ans:

                    batch_content.append({"q": q, "opts": opts, "ans": ans})



            if not batch_content: continue



            # Gọi Groq

            json_str = get_groq_response(batch_content)



            if json_str:

                try:

                    data = json.loads(json_str)

                    new_qs = []



                    if isinstance(data, dict):

                        # Ưu tiên tìm key "questions"

                        if "questions" in data and isinstance(data["questions"], list):

                            new_qs = data["questions"]

                        else:

                            # Tìm bất kỳ key nào chứa list

                            for key in data:

                                if isinstance(data[key], list):

                                    new_qs = data[key]

                                    break

                    elif isinstance(data, list):

                        new_qs = data



                    for q in new_qs:

                        if 'question' in q and 'options' in q:

                            f_out.write(json.dumps(q, ensure_ascii=False) + "\n")



                    f_out.flush()

                    os.fsync(f_out.fileno())



                except json.JSONDecodeError:

                    print("   ❌ Lỗi Parse JSON")



            # Model 8B rất nhanh, nghỉ cực ngắn là đủ

            time.sleep(0.5)



    print("\n" + "="*50)

    print("✅ HOÀN TẤT QUÁ TRÌNH!")

    print(f"📂 File kết quả: {OUTPUT_FILE}")



if __name__ == "__main__":

  main()

In [ ]:
!pip install -q transformers torch scikit-learn pandas accelerate

import os
import json
import torch
import random
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForMultipleChoice, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from google.colab import drive

drive.mount('/content/drive', force_remount=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Device: {DEVICE}")

# --- CẤU HÌNH ---
# Model cũ (V2)
OLD_MODEL_PATH = '/content/drive/MyDrive/DistilBERT_Retrained_QA_Model_V3_MultipleChoice'

# Dữ liệu
FILE_ORIGINAL = '/content/drive/MyDrive/data MCQ 9K.jsonl'
FILE_GENERATED = '/content/drive/MyDrive/generated_data_v3.jsonl'

# Model mới (V3)
NEW_MODEL_OUTPUT_DIR = '/content/drive/MyDrive/DistilBERT_Retrained_QA_Model_V4_MultipleChoice'

BATCH_SIZE = 16
EPOCHS = 4
LR = 3e-5

if not os.path.exists(NEW_MODEL_OUTPUT_DIR):
    os.makedirs(NEW_MODEL_OUTPUT_DIR)

# ============================================================
# PHẦN 1: HÀM ĐỌC DỮ LIỆU
# ============================================================
def load_data(file_path, tag="Data"):
    print(f"📂 Reading {tag}: {os.path.basename(file_path)}...")
    data = []

    if not os.path.exists(file_path):
        print(f"   ⚠️ File not found: {file_path}")
        return []

    count = 0
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                if line.strip().startswith('['):
                    items = json.loads(line)
                    if isinstance(items, list):
                        for obj in items:
                            if process_item(obj, data): count += 1
                else:
                    obj = json.loads(line)
                    if process_item(obj, data): count += 1
            except: continue

    print(f"   -> Loaded: {count}")
    return data

def process_item(obj, data_list):
    try:
        q = obj.get('question')
        opts = obj.get('options') or obj.get('choices') or obj.get('raw_options')
        ans = obj.get('answer') or obj.get('raw_answer') or obj.get('correct_answer') or obj.get('true_answer') or obj.get('label')

        if q and opts and ans is not None:
            choices = []
            answer_idx = -1

            if isinstance(opts, dict):
                sorted_k = sorted(opts.keys())
                choices = [str(opts[k]) for k in sorted_k]
                if isinstance(ans, str) and ans in sorted_k: answer_idx = sorted_k.index(ans)
            elif isinstance(opts, list):
                choices = [str(o) for o in opts]
                if isinstance(ans, str) and len(ans) == 1:
                    char_map = {'A':0, 'B':1, 'C':2, 'D':3}
                    answer_idx = char_map.get(ans.upper(), -1)
                elif isinstance(ans, int):
                    answer_idx = ans

            if len(choices) >= 2 and answer_idx != -1 and answer_idx < 4:
                while len(choices) < 4: choices.append("-")
                choices = choices[:4]

                data_list.append({
                    "question": str(q),
                    "choices": choices,
                    "label": int(answer_idx)
                })
                return True
    except: pass
    return False

# ============================================================
# PHẦN 2: CHUẨN BỊ TRAIN
# ============================================================
print("\n" + "="*40)
print("🥣 PREPARING DATA")
print("="*40)

data_orig = load_data(FILE_ORIGINAL, "Original")
data_gen = load_data(FILE_GENERATED, "Generated")

final_train_data = data_orig + data_gen
random.shuffle(final_train_data)

print(f"\n📚 Total samples: {len(final_train_data)}")

if len(final_train_data) == 0:
    import sys; sys.exit("❌ No data found!")

print(f"\n📥 Loading Model: {OLD_MODEL_PATH}")
try:
    tokenizer = AutoTokenizer.from_pretrained(OLD_MODEL_PATH)
    model = AutoModelForMultipleChoice.from_pretrained(OLD_MODEL_PATH).to(DEVICE)
    print("✅ Loaded Pre-trained Model")
except:
    print("⚠️ Pre-trained not found. Loading Base DistilBERT...")
    tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
    model = AutoModelForMultipleChoice.from_pretrained("distilbert-base-uncased").to(DEVICE)

class QADataset(Dataset):
    def __init__(self, tokenizer, data_list, max_len=256):
        self.tokenizer = tokenizer
        self.data = data_list
        self.max_len = max_len

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        inputs = self.tokenizer(
            [item["question"]] * 4,
            item["choices"],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids": inputs["input_ids"],
            "attention_mask": inputs["attention_mask"],
            "labels": torch.tensor(item["label"], dtype=torch.long)
        }

train_set, val_set = train_test_split(final_train_data, test_size=0.1, random_state=42)
train_loader = DataLoader(QADataset(tokenizer, train_set, 256), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(QADataset(tokenizer, val_set, 256), batch_size=BATCH_SIZE, shuffle=False)

optimizer = AdamW(model.parameters(), lr=LR)
scheduler = get_linear_schedule_with_warmup(optimizer, 0, len(train_loader) * EPOCHS)

# ============================================================
# PHẦN 3: HUẤN LUYỆN
# ============================================================
print("\n" + "="*40)
print("🚀 START TRAINING")
print("="*40)

best_loss = float('inf')

for epoch in range(EPOCHS):
    model.train()
    print(f"\nEpoch {epoch + 1}/{EPOCHS}")

    prog_bar = tqdm(train_loader, desc="Training")
    total_loss = 0

    for batch in prog_bar:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)

        input_ids = input_ids.view(-1, 4, input_ids.shape[-1])
        attention_mask = attention_mask.view(-1, 4, attention_mask.shape[-1])

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        prog_bar.set_postfix({'loss': loss.item()})

    model.eval()
    val_loss = 0; correct = 0; total = 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)

            input_ids = input_ids.view(-1, 4, input_ids.shape[-1])
            attention_mask = attention_mask.view(-1, 4, attention_mask.shape[-1])

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            val_loss += outputs.loss.item()
            preds = torch.argmax(outputs.logits, dim=1)
            correct += (preds == labels).sum().item()
            total += len(labels)

    avg_loss = val_loss / len(val_loader)
    acc = correct / total
    print(f"   -> Val Loss: {avg_loss:.4f} | Val Accuracy: {acc:.4f}")

    if avg_loss < best_loss:
        best_loss = avg_loss
        print(f"🔥 Saving Best Model to: {NEW_MODEL_OUTPUT_DIR}")
        model.save_pretrained(NEW_MODEL_OUTPUT_DIR)
        tokenizer.save_pretrained(NEW_MODEL_OUTPUT_DIR)

print("\n🏆 TRAINING COMPLETED!")

In [ ]:
# ==============================================================================
# SCRIPT CHẤM THI (OUTPUT FILE NAME CHUẨN ĐỊNH DẠNG)
# ==============================================================================

!pip install -q transformers datasets torch accelerate scikit-learn pandas tqdm

import os
import json
import torch
import gc
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from google.colab import drive
from tqdm import tqdm

# 1. Kết nối Drive & GPU
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    print(f"🚀 GPU Sẵn sàng: {torch.cuda.get_device_name(0)}")
else:
    DEVICE = torch.device('cpu')
    print("⚠️ CẢNH BÁO: Đang chạy CPU (Rất chậm)")

# ==============================================================================
# ⚙️ CẤU HÌNH (BẠN CHỈ CẦN SỬA Ở ĐÂY)
# ==============================================================================

# 1. Đường dẫn đến Model cần chấm
MODEL_PATH = "/content/drive/MyDrive/DistilBERT_Retrained_QA_Model_V4"

# 2. Tên phiên bản ngắn gọn (Để đặt tên file giống trong ảnh)
# Ví dụ: "V1", "V2", "QA_Model", "V5_HardMining"...
MODEL_TAG = "V4"

# 3. File đề thi
INPUT_FILE = '/content/drive/MyDrive/data MCQ 9K.jsonl'

# 4. Thư mục lưu kết quả
OUTPUT_DIR = '/content/drive/MyDrive/DistilBERT/Benchmark_Wrong_Cases'

# ------------------------------------------------------------------------------
# Tự động tạo tên file chuẩn: wrong_answers_{TAG}.jsonl
OUTPUT_FILE_NAME = f"wrong_answers_{MODEL_TAG}.jsonl"
ERROR_FILE_PATH = os.path.join(OUTPUT_DIR, OUTPUT_FILE_NAME)

# Tạo thư mục nếu chưa có
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

print(f"🔹 Đang chấm thi cho phiên bản: {MODEL_TAG}")
print(f"🔹 File kết quả sẽ là: {OUTPUT_FILE_NAME}") # Kiểm tra xem tên này khớp ảnh chưa

# ============================================================
# LOAD MODEL & TOKENIZER
# ============================================================
print(f"\n📥 Đang tải Model...")

try:
    # Load Tokenizer gốc
    tokenizer = AutoTokenizer.from_pretrained("distilbert-base-multilingual-cased")

    # Load Model
    if not os.path.exists(MODEL_PATH):
        print(f"❌ Lỗi: Không tìm thấy folder model tại {MODEL_PATH}")
        import sys; sys.exit()

    model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH).to(DEVICE)
    model.eval()
    print(f"✅ Model đã sẵn sàng!")
except Exception as e:
    print(f"❌ Lỗi load model: {e}")
    import sys; sys.exit()

# ============================================================
# BẮT ĐẦU CHẤM THI
# ============================================================
print(f"🚀 Bắt đầu làm bài thi...")

y_true = []
y_pred = []
wrong_cases = []

# Xóa file cũ nếu tồn tại để ghi mới (tránh ghi đè/nối thêm)
if os.path.exists(ERROR_FILE_PATH):
    os.remove(ERROR_FILE_PATH)

with open(INPUT_FILE, 'r', encoding='utf-8') as f:
    lines = f.readlines()

    for line in tqdm(lines, desc=f"Predicting {MODEL_TAG}"):
        if not line.strip(): continue
        try:
            item = json.loads(line)

            # Parse dữ liệu
            q = item.get('question') or item.get('q')
            opts = item.get('options') or item.get('opts')
            ans = item.get('answer') or item.get('correct_answer') or item.get('ans')

            if not q or not opts or not ans: continue
            if not isinstance(opts, dict): continue

            # Logic dự đoán
            best_score = -float('inf')
            pred_label = list(opts.keys())[0]

            for label, text in opts.items():
                inputs = tokenizer(q, text, return_tensors="pt", truncation=True, max_length=256).to(DEVICE)
                with torch.no_grad():
                    outputs = model(**inputs)
                    if outputs.logits.shape[1] > 1:
                        score = outputs.logits[0][1].item()
                    else:
                        score = outputs.logits[0][0].item()

                if score > best_score:
                    best_score = score
                    pred_label = label

            y_true.append(ans)
            y_pred.append(pred_label)

            # Lưu câu sai
            if pred_label != ans:
                wrong_cases.append({
                    "question": q,
                    "options": opts,
                    "correct_answer": ans,
                    "model_prediction": pred_label,
                    "model_source": MODEL_TAG
                })

        except: continue

# Dọn dẹp RAM
del model
torch.cuda.empty_cache()
gc.collect()

# ============================================================
# BÁO CÁO KẾT QUẢ
# ============================================================
if len(y_true) > 0:
    acc = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)

    print("\n" + "="*50)
    print(f"🏆 KẾT QUẢ PHIÊN BẢN: {MODEL_TAG}")
    print("="*50)
    print(f"🔹 Tổng số câu: {len(y_true)}")
    print(f"✅ Số câu ĐÚNG: {len(y_true) - len(wrong_cases)}")
    print(f"❌ Số câu SAI:  {len(wrong_cases)}")
    print("-" * 30)
    print(f"🎯 ACCURACY:    {acc*100:.2f}%")
    print(f"🎯 PRECISION:   {precision*100:.2f}%")
    print(f"🎯 RECALL:      {recall*100:.2f}%")
    print(f"🎯 F1-SCORE:    {f1*100:.2f}%")
    print("="*50)

    # Lưu file lỗi
    if wrong_cases:
        with open(ERROR_FILE_PATH, 'w', encoding='utf-8') as f:
            for case in wrong_cases:
                f.write(json.dumps(case, ensure_ascii=False) + '\n')
        print(f"\n📂 Đã lưu danh sách câu sai tại:\n   -> {ERROR_FILE_PATH}")
    else:
        print(f"\n🎉 Tuyệt vời! Phiên bản {MODEL_TAG} không sai câu nào.")
else:
    print("❌ Không đọc được dữ liệu nào từ file input.")

In [ ]:

# ==============================================================================

# TOOL SINH DỮ LIỆU GROQ (PHIÊN BẢN TỐC ĐỘ CAO - 8B INSTANT)

# ==============================================================================



!pip install -q groq tqdm



import os

import json

import time

from groq import Groq, RateLimitError, APIError, BadRequestError

from google.colab import drive

from tqdm import tqdm



# 1. Kết nối Drive

if not os.path.exists('/content/drive'):

    drive.mount('/content/drive', force_remount=True)



# --- CẤU HÌNH ---

GROQ_API_KEY = "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"



# 🔥 QUAN TRỌNG: CHUYỂN SANG 8B ĐỂ CÓ QUOTA CAO HƠN

GROQ_MODEL = "qwen/qwen3-32b"



INPUT_FILE = '/content/drive/MyDrive/DistilBERT/Benchmark_Wrong_Cases/wrong_answers_QA_Model_V4.jsonl'

OUTPUT_FILE = '/content/drive/MyDrive/DistilBERT/Generated_data_distil/generated_data_groq_distil_v4.jsonl'



# 🔥 GIẢM BATCH XUỐNG 1 ĐỂ AN TOÀN TUYỆT ĐỐI VỚI RATE LIMIT

BATCH_SIZE = 5

NUM_VARIANTS = 4



client = Groq(api_key=GROQ_API_KEY)



# ============================================================

# HÀM GỌI API

# ============================================================

def get_groq_response(batch_content):



    system_prompt = "You are an Expert Exam Creator. You output JSON only."



    # Prompt được tối ưu cho 8B

    user_prompt = f"""

    Task: Create exactly {NUM_VARIANTS} NEW multiple-choice variants for the input question.



    INPUT:

    {json.dumps(batch_content, ensure_ascii=False)}



    OUTPUT JSON FORMAT:

    {{

      "questions": [

        {{ "question": "...", "options": {{ "A": "...", "B": "...", "C": "...", "D": "..." }}, "answer": "A" }},

        ...

      ]

    }}

    STRICTLY JSON. NO EXPLANATION.

    """



    max_retries = 10

    for attempt in range(max_retries):

        try:

            chat_completion = client.chat.completions.create(

                messages=[

                    {"role": "system", "content": system_prompt},

                    {"role": "user", "content": user_prompt}

                ],

                model=GROQ_MODEL,

                response_format={"type": "json_object"},

                temperature=0.6,

                max_tokens=2048 # Giới hạn output để tiết kiệm TPM

            )

            return chat_completion.choices[0].message.content



        except RateLimitError:

            wait_time = 10 + (attempt * 5) # Chờ tăng dần: 10s, 15s, 20s...

            print(f"\r   ⏳ Hết Quota 8B. Đang nghỉ {wait_time}s... (Lần {attempt+1})", end="")

            time.sleep(wait_time)



        except (APIError, BadRequestError) as e:

            print(f"\n   ⚠️ Lỗi API: {e}. Thử lại sau 2s...")

            time.sleep(2)

        except Exception as e:

            print(f"\n   ⚠️ Lỗi lạ: {e}")

            time.sleep(2)



    return None



# ============================================================

# MAIN LOOP

# ============================================================

def main():

    print(f"🚀 BẮT ĐẦU VỚI MODEL: {GROQ_MODEL} (Tốc độ cao)")



    if not os.path.exists(INPUT_FILE):

        print(f"❌ Lỗi: Không tìm thấy file {INPUT_FILE}")

        return



    input_data = []

    with open(INPUT_FILE, 'r', encoding='utf-8') as f:

        for line in f:

            try: input_data.append(json.loads(line))

            except: continue



    total_input = len(input_data)

    print(f"📚 Tổng số câu sai cần xử lý: {total_input}")



    # Resume Logic

    start_idx = 0

    if os.path.exists(OUTPUT_FILE):

        with open(OUTPUT_FILE, 'r') as f:

            existing_lines = sum(1 for _ in f)



        # Tính toán lại vị trí resume

        processed_items = existing_lines // NUM_VARIANTS

        start_idx = processed_items



        # Căn chỉnh batch

        start_idx = (start_idx // BATCH_SIZE) * BATCH_SIZE

        print(f"🔄 File cũ có {existing_lines} dòng. Tiếp tục từ câu gốc thứ {start_idx}...")



    print("\n⚡ Đang chạy...")



    with open(OUTPUT_FILE, 'a', encoding='utf-8') as f_out:



        # Cắt danh sách input

        remaining_data = input_data[start_idx:]

        batches = [remaining_data[i:i + BATCH_SIZE] for i in range(0, len(remaining_data), BATCH_SIZE)]



        for batch in tqdm(batches, desc="Tiến độ"):



            # Chuẩn bị Input

            batch_content = []

            for item in batch:

                q = item.get('question') or item.get('q')

                opts = item.get('options') or item.get('raw_options') or item.get('opts')

                ans = item.get('answer') or item.get('raw_answer') or item.get('correct_answer')

                if q and opts and ans:

                    batch_content.append({"q": q, "opts": opts, "ans": ans})



            if not batch_content: continue



            # Gọi Groq

            json_str = get_groq_response(batch_content)



            if json_str:

                try:

                    data = json.loads(json_str)

                    new_qs = []



                    if isinstance(data, dict):

                        # Ưu tiên tìm key "questions"

                        if "questions" in data and isinstance(data["questions"], list):

                            new_qs = data["questions"]

                        else:

                            # Tìm bất kỳ key nào chứa list

                            for key in data:

                                if isinstance(data[key], list):

                                    new_qs = data[key]

                                    break

                    elif isinstance(data, list):

                        new_qs = data



                    for q in new_qs:

                        if 'question' in q and 'options' in q:

                            f_out.write(json.dumps(q, ensure_ascii=False) + "\n")



                    f_out.flush()

                    os.fsync(f_out.fileno())



                except json.JSONDecodeError:

                    print("   ❌ Lỗi Parse JSON")



            # Model 8B rất nhanh, nghỉ cực ngắn là đủ

            time.sleep(0.5)



    print("\n" + "="*50)

    print("✅ HOÀN TẤT QUÁ TRÌNH!")

    print(f"📂 File kết quả: {OUTPUT_FILE}")



if __name__ == "__main__":

  main()

In [ ]:
# ==============================================================================
# SCRIPT TRAIN NÂNG CẤP MODEL (INCREMENTAL TRAINING)
# Dùng để train từ Model cũ -> Model mới (kết hợp dữ liệu Gốc + Dữ liệu Sinh)
# ==============================================================================

!pip install -q transformers datasets accelerate scikit-learn pandas

import os
import json
import torch
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from google.colab import drive
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

# 1. Kết nối Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Thiết bị training: {DEVICE}")

# ==============================================================================
# ⚙️ CẤU HÌNH (BẠN CHỈ CẦN SỬA 4 DÒNG NÀY CHO MỖI LẦN TRAIN)
# ==============================================================================

# 1. Đường dẫn Model CŨ (Model nền tảng để học tiếp)
BASE_MODEL_PATH = '/content/drive/MyDrive/DistilBERT_Retrained_QA_Model_V4'

# 2. Đường dẫn lưu Model MỚI (Kết quả sau khi train xong)
NEW_MODEL_OUTPUT_DIR = '/content/drive/MyDrive/DistilBERT_Retrained_QA_Model_V5'

# 3. File dữ liệu GỐC (Kiến thức nền)
FILE_DATA_ORIGIN = '/content/drive/MyDrive/data MCQ 9K.jsonl'

# 4. File dữ liệu BỔ SUNG (Dữ liệu sinh / Dữ liệu sửa lỗi)
FILE_DATA_AUGMENTED = '/content/drive/MyDrive/DistilBERT/Generated_data_distil/generated_data_groq_distil_v4.jsonl'

# ==============================================================================

# Tự động tạo thư mục đầu ra nếu chưa có
if not os.path.exists(NEW_MODEL_OUTPUT_DIR):
    os.makedirs(NEW_MODEL_OUTPUT_DIR)

print(f"🔹 BASE MODEL:  {os.path.basename(BASE_MODEL_PATH)}")
print(f"🔹 TARGET MODEL: {os.path.basename(NEW_MODEL_OUTPUT_DIR)}")

# ============================================================
# PHẦN 1: TẢI & TRỘN DỮ LIỆU
# ============================================================
print("\n" + "="*50)
print("🟢 BƯỚC 1: CHUẨN BỊ DỮ LIỆU (MIX DATA)")
print("="*50)

# Load Tokenizer từ Base Model
print(f"📥 Đang tải Tokenizer...")
try:
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_PATH)
except:
    print("⚠️ Không tìm thấy Tokenizer ở model cũ, tải bản gốc DistilBERT...")
    tokenizer = AutoTokenizer.from_pretrained("distilbert-base-multilingual-cased")

def load_data_generic(file_path, tag=""):
    data_pairs = []
    if not os.path.exists(file_path):
        print(f"⚠️ Cảnh báo: Không tìm thấy file {tag}: {file_path}")
        return []

    print(f"   -> Đang đọc {tag}: {os.path.basename(file_path)}")
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip(): continue
            try:
                item = json.loads(line)
                # Parse dữ liệu linh hoạt
                q = item.get('question') or item.get('q')
                opts = item.get('options') or item.get('opts')
                ans = item.get('answer') or item.get('correct_answer') or item.get('ans')

                if not q or not opts or not ans: continue

                if isinstance(opts, dict):
                    for label, text in opts.items():
                        is_correct = 1 if label == ans else 0
                        input_text = f"{q} {tokenizer.sep_token} {text}"
                        data_pairs.append({
                            "text": input_text,
                            "label": is_correct
                        })
            except: continue
    print(f"      + Số lượng: {len(data_pairs)} mẫu")
    return data_pairs

# Load dữ liệu
data_1 = load_data_generic(FILE_DATA_ORIGIN, "DATA GỐC")
data_2 = load_data_generic(FILE_DATA_AUGMENTED, "DATA BỔ SUNG")

if not data_1 and not data_2:
    import sys; sys.exit("❌ Lỗi: Không có dữ liệu nào để train!")

# Trộn dữ liệu
full_data = data_1 + data_2
print(f"✅ TỔNG DỮ LIỆU HUẤN LUYỆN: {len(full_data)} mẫu.")

# Tạo DataFrame & Xáo trộn (Shuffle)
df = pd.DataFrame(full_data)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Chia tập 70-15-15
train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42, stratify=df['label'])
eval_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])

print(f"📊 Dataset Split:")
print(f"   - Train: {len(train_df)}")
print(f"   - Eval:  {len(eval_df)}")
print(f"   - Test:  {len(test_df)}")

# Tạo Dataset object
train_dataset = Dataset.from_pandas(train_df)
eval_dataset = Dataset.from_pandas(eval_df)
test_dataset = Dataset.from_pandas(test_df)

# Tokenize
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=256, padding="max_length")

print("⏳ Đang Tokenize...")
tokenized_train = train_dataset.map(preprocess_function, batched=True)
tokenized_eval = eval_dataset.map(preprocess_function, batched=True)
tokenized_test = test_dataset.map(preprocess_function, batched=True)

# ============================================================
# PHẦN 2: TRAINING (FINE-TUNE)
# ============================================================
print("\n" + "="*50)
print("🔵 BƯỚC 2: TIẾN HÀNH TRAINING")
print("="*50)

# Load Model Cũ
print(f"📥 Đang tải weights từ Base Model: {BASE_MODEL_PATH}")
model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL_PATH, num_labels=2).to(DEVICE)

training_args = TrainingArguments(
    output_dir="./results_incremental",
    learning_rate=1e-5,              # LR thấp để tinh chỉnh (Fine-tuning)
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,              # Số vòng lặp
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none"
)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return {"accuracy": (predictions == labels).mean()}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()

# ============================================================
# PHẦN 3: LƯU KẾT QUẢ
# ============================================================
print("\n" + "="*50)
print("🟣 BƯỚC 3: ĐÁNH GIÁ & LƯU MODEL MỚI")
print("="*50)

# Đánh giá trên tập Test độc lập
test_results = trainer.evaluate(tokenized_test)
acc = test_results['eval_accuracy'] * 100
print(f"🏆 Final Test Accuracy: {acc:.2f}%")

# Lưu Model
print(f"💾 Đang lưu model mới vào: {NEW_MODEL_OUTPUT_DIR}")
model.save_pretrained(NEW_MODEL_OUTPUT_DIR)
tokenizer.save_pretrained(NEW_MODEL_OUTPUT_DIR)

print("\n🎉 HOÀN TẤT! Model đã sẵn sàng để sử dụng.")

In [ ]:
# ==============================================================================
# GENERIC EVALUATION SCRIPT (CHẤM THI DISTILBERT MULTIPLE CHOICE)
# ==============================================================================

# 1. Cài đặt thư viện
# !pip install -q transformers torch scikit-learn pandas accelerate

import os
import json
import torch
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForMultipleChoice
from sklearn.metrics import accuracy_score, classification_report
from tqdm import tqdm
from google.colab import drive

# 2. Kết nối Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive', force_remount=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[INFO] Device: {DEVICE}")

# --- CẤU HÌNH (SỬA Ở ĐÂY CHO DISTILBERT) ---

# 1. Đường dẫn Model DistilBERT cần chấm
CURRENT_MODEL_PATH = '/content/drive/MyDrive/DistilBERT_FineTuned_QA_Model_V5_MultipleChoice'

# 2. File dữ liệu đề thi (Test Set - JSONL)
TEST_DATA_PATH = '/content/drive/MyDrive/data MCQ 9K.jsonl'

# 3. Nơi lưu kết quả (File chứa các câu làm sai) -> ĐÃ CẬP NHẬT THEO YÊU CẦU
OUTPUT_WRONG_ANSWERS = '/content/drive/MyDrive/Benchmark_Wrong_Cases/wrong_answers_V5.jsonl'

# Tham số khác
BATCH_SIZE = 16
MAX_LEN = 256

# ============================================================
# 1. HÀM ĐỌC DỮ LIỆU
# ============================================================
def load_data(file_path):
    print(f"[INFO] Reading data from: {os.path.basename(file_path)}")
    data = []
    if not os.path.exists(file_path):
        print(f"[ERROR] File not found: {file_path}")
        return []

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                obj = json.loads(line)
                if 'question' in obj and 'options' in obj and 'answer' in obj:
                    opts = obj['options']
                    sorted_k = sorted(opts.keys()) # A, B, C, D

                    if obj['answer'] in sorted_k:
                        data.append({
                            "question": obj['question'],
                            "choices": [opts[k] for k in sorted_k],
                            "label": sorted_k.index(obj['answer']),
                            "raw_options": opts,
                            "raw_answer": obj['answer']
                        })
            except: continue

    print(f"[INFO] Loaded {len(data)} valid samples.")
    return data

# ============================================================
# 2. DATASET CLASS
# ============================================================
class EvalDataset(Dataset):
    def __init__(self, tokenizer, data_list, max_len=256):
        self.tokenizer = tokenizer
        self.data = data_list
        self.max_len = max_len

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        inputs = self.tokenizer(
            [item["question"]] * len(item["choices"]),
            item["choices"],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids": inputs["input_ids"],
            "attention_mask": inputs["attention_mask"],
            "labels": torch.tensor(item["label"], dtype=torch.long)
        }

# ============================================================
# 3. HÀM CHẤM THI (MAIN EVALUATION)
# ============================================================
def run_evaluation():
    # Load Data
    all_data = load_data(TEST_DATA_PATH)
    if not all_data: return

    # Load Model & Tokenizer
    print(f"\n[INFO] Loading model from: {CURRENT_MODEL_PATH}")
    try:
        try:
            tokenizer = AutoTokenizer.from_pretrained(CURRENT_MODEL_PATH)
        except:
            print("[WARN] Local tokenizer not found. Using fallback: 'distilbert-base-multilingual-cased'")
            tokenizer = AutoTokenizer.from_pretrained("distilbert-base-multilingual-cased", use_fast=True)

        model = AutoModelForMultipleChoice.from_pretrained(CURRENT_MODEL_PATH).to(DEVICE)
        model.eval()
        print("[INFO] Model loaded successfully.")

    except Exception as e:
        print(f"[ERROR] Critical failure loading model: {e}")
        return

    # Prepare Loader
    dataset = EvalDataset(tokenizer, all_data, max_len=MAX_LEN)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

    print("\n[INFO] Starting Scoring Process for DistilBERT...")

    all_preds = []
    all_labels = []
    wrong_cases = []
    batch_start_idx = 0

    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            targets = labels.cpu().numpy()

            all_preds.extend(preds)
            all_labels.extend(targets)

            # Ghi lại câu sai
            for i in range(len(preds)):
                if preds[i] != targets[i]:
                    original_item = all_data[batch_start_idx + i]
                    char_map = ['A', 'B', 'C', 'D']
                    pred_char = char_map[preds[i]] if preds[i] < 4 else "Unknown"

                    wrong_cases.append({
                        "question": original_item['question'],
                        "options": original_item['raw_options'],
                        "correct_answer": original_item['raw_answer'],
                        "model_prediction": pred_char
                    })

            batch_start_idx += len(preds)

    # Calculate Metrics
    acc = accuracy_score(all_labels, all_preds)

    print("\n" + "="*40)
    print(f"REPORT RESULTS")
    print("="*40)
    print(f"Model Path:      {os.path.basename(CURRENT_MODEL_PATH)}")
    print(f"Total Questions: {len(all_data)}")
    print(f"Correct:         {len(all_data) - len(wrong_cases)}")
    print(f"Wrong:           {len(wrong_cases)}")
    print(f"Accuracy:        {acc:.4f} ({acc*100:.2f}%)")

    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds, target_names=['A', 'B', 'C', 'D'], zero_division=0))

    # Save Wrong Answers
    if wrong_cases:
        # Tự động tạo thư mục nếu chưa tồn tại (Benchmark_Wrong_Cases)
        output_dir = os.path.dirname(OUTPUT_WRONG_ANSWERS)
        if output_dir and not os.path.exists(output_dir):
            print(f"[INFO] Creating directory: {output_dir}")
            os.makedirs(output_dir)

        with open(OUTPUT_WRONG_ANSWERS, 'w', encoding='utf-8') as f:
            for item in wrong_cases:
                f.write(json.dumps(item, ensure_ascii=False) + '\n')
        print(f"\n[SAVE] Saved {len(wrong_cases)} wrong answers to: {OUTPUT_WRONG_ANSWERS}")
    else:
        print("\n[INFO] Perfect Score! No wrong answers file generated.")

if __name__ == "__main__":
    run_evaluation()